# Bloque 1. Importaciones y Configuración.

In [1256]:
import re

# --- MAPA DE PIEZAS ---
PIECE_MAP = {
    'A': 'arqueros', 'B': 'escorpion', 'C': 'c_ligera', 
    'D': 'dragon',   'E': 'elefante',  'F': 'fortaleza',
    'H': 'chusma',   'L': 'lanceros',  'M': 'montana',
    'P': 'c_pesada', 'R': 'rey',       'T': 'trabuquete'
}

# --- CONFIGURACIÓN DE CANTIDADES ---
# Ajusta estos números según las reglas oficiales de Cotadrez
LIMITES_INICIALES = {
    'rey': 1,
    'fortaleza': 1,
    'montana': 3,       # Por lo que vi en el log
    'lanceros': 3,
    'arqueros': 3,
    'c_ligera': 3,
    'c_pesada': 2,
    'chusma': 3,
    'dragon': 1,
    'elefante': 2,
    'escorpion': 1,
    'trabuquete': 1
}

# Mapeo inverso para pintar el tablero (Nombre -> Letra)
TYPE_TO_CHAR = {v: k for k, v in PIECE_MAP.items()}

# Colores ANSI para la terminal
C_RESET  = "\033[0m"
C_GRAY   = "\033[90m"      # Gris (Vacío)
C_BLUE   = "\033[94m"      # Azul (Agua)
C_RED    = "\033[91m"      # Rojo (Ejército Rojo)
C_BLACK  = "\033[1;37m"    # Blanco Negrita (Para el Ejército Negro en fondo oscuro)
# Nota: Si usas fondo claro, cambia C_BLACK a "\033[30m"

# --- FUNCIÓN AUXILIAR (SIGN) ---
def sign(x):
    if x > 0: return 1
    if x < 0: return -1
    return 0

# Bloque 2. Funciones de ataque y amenazas.

In [1257]:
def can_unit_attack(piece_type, r1, c1, r2, c2, army, board, terrain, ignore_pos=None):
    """Valida si una unidad puede atacar/capturar una casilla."""
    dr = abs(r2 - r1)
    dc = abs(c2 - c1)

    # 1. UNIDADES SIMPLES
    if piece_type == 'rey': return dr <= 1 and dc <= 1
    if piece_type == 'lanceros' or piece_type == 'chusma': return (dr + dc == 1)

    # 2. ARQUEROS
    if piece_type == 'arqueros':
        if dr != dc: return False
        if dr == 1: return True
        if dr == 2:
            mid_r, mid_c = (r1 + r2) // 2, (c1 + c2) // 2
            is_mid_empty = (board[mid_r][mid_c] is None) or \
                           (ignore_pos and mid_r == ignore_pos['r'] and mid_c == ignore_pos['c'])
            return is_mid_empty and terrain[mid_r][mid_c] != 'water'
        return False

    # 3. CABALLERÍA LIGERA
    if piece_type == 'c_ligera':
        if not ((dr == 3 and dc == 1) or (dr == 1 and dc == 3)): return False
        
        def check_block(r, c):
            if r < 0 or r > 9 or c < 0 or c > 9: return True
            if ignore_pos and r == ignore_pos['r'] and c == ignore_pos['c']: return False
            if terrain[r][c] == 'water': return True
            o = board[r][c]
            return (o and (o['type'] == 'montana' or o['type'] == 'fortaleza'))

        # Ruta A (2+1)
        diags2 = [(2, 2), (2, -2), (-2, 2), (-2, -2)]
        for kr, kc in diags2:
            if abs(r2 - (r1 + kr)) == 1 and abs(c2 - (c1 + kc)) == 1:
                knee_r, knee_c = r1 + kr, c1 + kc
                mid_r, mid_c = (r1 + knee_r) // 2, (c1 + knee_c) // 2
                if not check_block(mid_r, mid_c) and not check_block(knee_r, knee_c): return True

        # Ruta B (1+2)
        diags1 = [(1, 1), (1, -1), (-1, 1), (-1, -1)]
        for kr, kc in diags1:
            if abs(r2 - (r1 + kr)) == 2 and abs(c2 - (c1 + kc)) == 2:
                knee_r, knee_c = r1 + kr, c1 + kc
                mid_r, mid_c = (knee_r + r2) // 2, (knee_c + c2) // 2
                if not check_block(knee_r, knee_c) and not check_block(mid_r, mid_c): return True
        return False

    # 4. CABALLERÍA PESADA
    if piece_type == 'c_pesada':
        if not ((dr == 2 and dc == 1) or (dr == 1 and dc == 2)): return False
        
        def is_blocked(r, c):
            if r < 0 or r > 9 or c < 0 or c > 9: return True
            if terrain[r][c] == 'water': return True
            if ignore_pos and r == ignore_pos['r'] and c == ignore_pos['c']: return False
            o = board[r][c]
            if o:
                if o['type'] in ['montana', 'fortaleza']: return True
                if o['army'] != army: return True
            return False

        sr, sc = sign(r2 - r1), sign(c2 - c1)
        if dr == 2:
            return not (is_blocked(r1 + sr, c1) or is_blocked(r2, c1)) or \
                   not (is_blocked(r1, c2) or is_blocked(r1 + sr, c2))
        else:
            return not (is_blocked(r1, c1 + sc) or is_blocked(r1, c2)) or \
                   not (is_blocked(r2, c1) or is_blocked(r2, c1 + sc))

    # 5. RAYCAST (Dragón, Elefante, Armas)
    # Lógica simplificada de raycast para validación rápida
    sr, sc = sign(r2 - r1), sign(c2 - c1)
    
    # Validar dirección
    is_ortho = (r1 == r2 or c1 == c2)
    is_diag = (dr == dc)
    
    if piece_type == 'dragon': 
        if not (is_ortho or is_diag): return False
    elif piece_type == 'elefante':
        if not is_ortho: return False
    elif piece_type == 'trabuquete': # Disparo
        if not is_ortho: return False
    elif piece_type == 'escorpion': # Disparo
        if not is_diag: return False

    cr, cc = r1 + sr, c1 + sc
    enemies_in_path = 0
    last_enemy_pos = None

    while cr != r2 or cc != c2:
        # --- 🛡️ MODIFICACIÓN QUIRÚRGICA: PROTECCIÓN DE LÍMITES ---
        # Si el rayo se sale del tablero, detenemos la validación inmediatamente
        if not (0 <= cr < 10 and 0 <= cc < 10):
            return False 
        # ---------------------------------------------------------

        if ignore_pos and cr == ignore_pos['r'] and cc == ignore_pos['c']:
            cr += sr; cc += sc; continue
        
        obs = board[cr][cc]
        
        if piece_type == 'dragon':
            if obs and obs['type'] != 'montana': return False
        
        elif piece_type == 'elefante':
            if terrain[cr][cc] == 'water' or obs: return False
            
        elif piece_type in ['trabuquete', 'escorpion']: # Armas
            if obs and obs['type'] in ['montana', 'fortaleza']: return False
            if piece_type == 'escorpion' and obs:
                if obs['army'] != army: 
                    enemies_in_path += 1; last_enemy_pos = {'r':cr, 'c':cc}
                else: return False
        
        cr += sr; cc += sc

    if piece_type == 'escorpion' and enemies_in_path > 0:
        if enemies_in_path == 1:
            dist = abs(r2 - last_enemy_pos['r'])
            return (dist >= 1 and dist <= 3)
        return False

    return True

def count_total_threats(target_r, target_c, army, board, terrain):
    """Cuenta cuántos enemigos (army) pueden atacar target."""
    threats = 0
    for tr in range(10):
        for tc in range(10):
            piece = board[tr][tc]
            if piece and piece['army'] == army:
                if can_unit_attack(piece['type'], tr, tc, target_r, target_c, army, board, terrain):
                    threats += 1
    return threats

def get_attackers(target_r, target_c, victim_army, board, terrain):
    """Devuelve lista de atacantes enemigos."""
    attackers = []
    enemy_army = 'negro' if victim_army == 'rojo' else 'rojo'
    for r in range(10):
        for c in range(10):
            p = board[r][c]
            if p and p['army'] == enemy_army:
                if can_unit_attack(p['type'], r, c, target_r, target_c, enemy_army, board, terrain):
                    attackers.append({'r': r, 'c': c, 'type': p['type']})
    return attackers

# Bloque 3. Lógica principal de movimiento.

In [1258]:
def is_valid_move(piece_type, r1, c1, r2, c2, board, terrain):
    dr, dc = abs(r2 - r1), abs(c2 - c1)
    my_piece = board[r1][c1]
    target = board[r2][c2]

    # 0. REGLA ELEFANTE
    if my_piece and target and target['type'] == 'elefante' and target['army'] != my_piece['army']:
        if piece_type != 'dragon':
            threats = count_total_threats(r2, c2, my_piece['army'], board, terrain)
            if threats < 2: return False

    # 1. REY
    if piece_type == 'rey': return (dr <= 1 and dc <= 1) and (dr + dc > 0)

    # 2. LANCEROS / CHUSMA
    if piece_type in ['lanceros', 'chusma']: return (dr + dc == 1)

    # 3. DRAGÓN
    if piece_type == 'dragon':
        if not ((dr == dc) or (r1 == r2 or c1 == c2)): return False
        sr, sc = sign(r2 - r1), sign(c2 - c1)
        cr, cc = r1 + sr, c1 + sc
        while cr != r2 or cc != c2:
            obs = board[cr][cc]
            if obs and obs['type'] != 'montana': return False
            cr += sr; cc += sc
        return True

    # 4. ARQUEROS
    if piece_type == 'arqueros':
        if dr != dc: return False
        if dr == 1: return True
        if dr == 2:
            if target is None: return False # Solo captura saltando
            mid_r, mid_c = (r1 + r2) // 2, (c1 + c2) // 2
            return (board[mid_r][mid_c] is None and terrain[mid_r][mid_c] != 'water')
        return False

    # 5. ELEFANTE
    if piece_type == 'elefante':
        if r1 != r2 and c1 != c2: return False
        sr, sc = sign(r2 - r1), sign(c2 - c1)
        cr, cc = r1 + sr, c1 + sc
        while cr != r2 or cc != c2:
            if board[cr][cc] is not None or terrain[cr][cc] == 'water': return False
            cr += sr; cc += sc
        return True

    # 6. C. LIGERA y 7. C. PESADA
    # Reutilizamos la lógica de ataque que es idéntica geométricamente
    if piece_type in ['c_ligera', 'c_pesada']:
        return can_unit_attack(piece_type, r1, c1, r2, c2, my_piece['army'], board, terrain)

    # 8. TRABUQUETE (Movimiento)
    if piece_type == 'trabuquete':
        if target is None: return (dr == 1 and dc == 1) # Mover
        # Disparo: Ortogonal
        if r1 != r2 and c1 != c2: return False
        sr, sc = sign(r2 - r1), sign(c2 - c1)
        cr, cc = r1 + sr, c1 + sc
        while cr != r2 or cc != c2:
            o = board[cr][cc]
            if o and o['type'] in ['montana', 'fortaleza']: return False
            cr += sr; cc += sc
        return True

    # 9. ESCORPIÓN (Movimiento)
    if piece_type == 'escorpion':
        if target is None: return (dr + dc == 1) # Mover
        # Disparo: Diagonal (usa lógica compleja de can_unit_attack)
        return can_unit_attack('escorpion', r1, c1, r2, c2, my_piece['army'], board, terrain)

    return False

def is_simulated_move_safe(piece_type, r1, c1, r2, c2, army, board, terrain):
    """Verifica si el movimiento deja al Rey en Jaque (Suicidio)."""
    king_r, king_c = None, None
    if piece_type == 'rey': king_r, king_c = r2, c2
    else:
        for r in range(10):
            for c in range(10):
                p = board[r][c]
                if p and p['type'] == 'rey' and p['army'] == army:
                    king_r, king_c = r, c; break
            if king_r: break
    
    if king_r is None: return True # Sin rey no hay jaque (debug)

    # Simular
    orig_src = board[r1][c1]
    orig_dst = board[r2][c2]
    
    board[r1][c1] = None
    board[r2][c2] = orig_src # Nota: No simulamos relevo complejo aquí para brevedad
    
    threats = get_attackers(king_r, king_c, army, board, terrain)
    
    # Revertir
    board[r1][c1] = orig_src
    board[r2][c2] = orig_dst
    
    return len(threats) == 0

# Bloque 4. El motor del juego.

In [1259]:
class CotadrezGame:
    def __init__(self):
        self.board = [[None]*10 for _ in range(10)]
        self.terrain = [[None]*10 for _ in range(10)]
        self._init_water()
        
        # Configuración de Jugadores
        self.j1_color = 'rojo'
        self.j2_color = 'negro'
        self.turn_color = 'rojo' 
        self.game_phase = 'setup'
        
        self.fortress_bounds = {'rojo': None, 'negro': None}
        self.dungeons = {'rojo': [], 'negro': []}
        
        # --- 🛡️ CORRECCIÓN DE RESERVAS ---
        self.reserves = {'rojo': {}, 'negro': {}}
        
        # Inicializamos las reservas leyendo el diccionario de límites
        for nombre_pieza, cantidad in LIMITES_INICIALES.items():
            self.reserves['rojo'][nombre_pieza] = cantidad
            self.reserves['negro'][nombre_pieza] = cantidad

    def set_j1_color(self, c_raw):
        c = c_raw.lower()
        if "rojo" in c: self.j1_color, self.j2_color = 'rojo', 'negro'
        elif "negro" in c: self.j1_color, self.j2_color = 'negro', 'rojo'
        self.turn_color = self.j1_color
        print(f"⚙️  J1={self.j1_color.upper()} | J2={self.j2_color.upper()}")

    def _init_water(self):
        for r, c in [(3,1), (1,3)]:
            self.terrain[r][c] = self.terrain[r][9-c] = \
            self.terrain[9-r][c] = self.terrain[9-r][9-c] = 'water'

    def coord_to_index(self, s):
        try: return 10 - int(s[1:]), 'abcdefghij'.index(s[0].lower())
        except: return None, None

    def index_to_coord(self, r, c):
        if r is None or c is None: return "??"
        return f"{'abcdefghij'[c]}{10-r}"

    def fmt(self, r, c):
        return f"({r},{c}|{self.index_to_coord(r,c)})"

    def switch_turn(self):
        self.turn_color = 'negro' if self.turn_color == 'rojo' else 'rojo'
    
    def get_icon(self, c=None): return "🔴" if (c or self.turn_color) == 'rojo' else "⚫"

    def deploy_fortress_area(self, r1, c1, r2, c2):
        army = self.turn_color
        
        # 1. COMPROBACIÓN DE RESERVA
        if self.reserves[army].get('fortaleza', 0) <= 0:
            return print(f"❌ ILEGAL: No quedan fortalezas en la reserva de {army}")

        min_r, max_r = min(r1, r2), max(r1, r2)
        min_c, max_c = min(c1, c2), max(c1, c2)
        if (max_r - min_r) != 1 or (max_c - min_c) != 1: return print("❌ ILEGAL: 2x2")
        
        self.fortress_bounds[army] = {'min_r': min_r, 'max_r': max_r, 'min_c': min_c, 'max_c': max_c}
        for cr in [min_r, max_r]:
            for cc in [min_c, max_c]:
                self.board[cr][cc] = {'type': 'fortaleza', 'army': army}
        
        # 2. RESTAR DE LA RESERVA
        self.reserves[army]['fortaleza'] -= 1
        
        # print(f"{self.get_icon(army)} Fortaleza OK {min_r}-{max_r}, {min_c}-{max_c}")

    def deploy_unit(self, p_type, r, c):
        army = self.turn_color
        pos_str = self.fmt(r, c) # Usamos el nuevo formateador
        
        if self.board[r][c]: return print(f"⚠️ Ocupado en {pos_str}")
        
        # --- [BLOQUEO TOTAL DE AGUA] ---
        if self.terrain[r][c] == 'water':
            return print(f"❌ ILEGAL: No se puede desplegar {p_type.upper()} en agua {pos_str}")

        if p_type == 'montana':
            self.board[r][c] = {'type': p_type, 'army': army}
            return True
        
        b = self.fortress_bounds[army]
        if not b: return print("❌ Falta Fortaleza")
        
        # Validación de anillo
        if max(max(0, b['min_r']-r, r-b['max_r']), max(0, b['min_c']-c, c-b['max_c'])) != 1:
            return print(f"❌ {p_type} en {pos_str} fuera de anillo")
        
        self.board[r][c] = {'type': p_type, 'army': army}
        if self.reserves[army].get(p_type, 0) > 0: self.reserves[army][p_type] -= 1
        
        # Print actualizado (quitamos el print manual aquí, lo delegamos al parser o usamos un return limpio)
        # Para mantener coherencia con tu log actual, imprimimos confirmación:
        print(f"🛡️  {self.get_icon()} {p_type.upper()} -> {pos_str}")
        return True

    def rescue_piece(self, p_type):
        self.reserves[self.turn_color][p_type] = self.reserves[self.turn_color].get(p_type, 0) + 1

    def execute_shot(self, r1, c1, r2, c2):
        target = self.board[r2][c2]
        if target:
            self.dungeons[self.turn_color].append(target)
            self.board[r2][c2] = None
            print(f"🔥 {self.get_icon()} DISPARO {self.fmt(r1,c1)} -> {self.fmt(r2,c2)}")

    # --- AQUÍ ESTÁ LA FUNCIÓN QUE FALTABA (apply_move) ---
    def apply_move_internal(self, r1, c1, r2, c2):
        """Ejecuta el movimiento validando reglas."""
        piece = self.board[r1][c1]
        
        # Validaciones de seguridad
        if not piece:
            print("❌ Origen vacío")
            return False
        
        # Validar legalidad (Usamos las funciones globales)
        if not is_valid_move(piece['type'], r1, c1, r2, c2, self.board, self.terrain):
             print(f"❌ Movimiento Ilegal (Geometría/Bloqueo)")
             # En un log real, podríamos querer forzarlo, pero aquí avisamos
             return False 
        
        if not is_simulated_move_safe(piece['type'], r1, c1, r2, c2, piece['army'], self.board, self.terrain):
             print(f"❌ ILEGAL: El movimiento deja al Rey en Jaque")
             return False

        target = self.board[r2][c2]
        
        # Lógica de Captura / Relevo
        if target:
            if target['army'] == piece['army']: # Relevo
                self.board[r1][c1] = target
                self.board[r2][c2] = piece
                # AÑADIDO PRINT RELEVO
                print(f"🔄 {self.get_icon()} RELEVO {self.fmt(r1,c1)} -> {self.fmt(r2,c2)}")
                return
            else: # Captura
                self.dungeons[self.turn_color].append(target)
                # AÑADIDO PRINT CAPTURA (Opcional, o dejamos que el print de mover cubra ambos)
                print(f"⚔️ {self.get_icon()} CAPTURA {self.fmt(r1,c1)} -> {self.fmt(r2,c2)}")
        
        # Mover (Si no hubo return antes por relevo, llegamos aquí)
        self.board[r2][c2] = piece
        self.board[r1][c1] = None
        
        # AÑADIDO PRINT MOVIMIENTO (Solo si no fue captura, para no duplicar, o siempre)
        if not target:
             print(f"➡️ {self.get_icon()} MUEVE {self.fmt(r1,c1)} -> {self.fmt(r2,c2)}")
             return True

    def find_piece_candidates(self, p_type, dest_r, dest_c, is_capture):
        candidates = []
        for r in range(10):
            for c in range(10):
                p = self.board[r][c]
                if p and p['type'] == p_type and p['army'] == self.turn_color:
                    if is_valid_move(p_type, r, c, dest_r, dest_c, self.board, self.terrain):
                        candidates.append({'r': r, 'c': c})
        return candidates
    
    # --- VISUALIZACIÓN (LA GUINDA 🍒) ---
    def render(self):
        print("\n   " + " ".join("abcdefghij")) # Cabecera Columnas
        
        for r in range(10):
            # Número de fila (izquierda)
            line = f"{10-r:2} "
            
            for c in range(10):
                piece = self.board[r][c]
                terr = self.terrain[r][c]
                
                # 1. CASILLA CON PIEZA
                if piece:
                    char = TYPE_TO_CHAR.get(piece['type'], '?')
                    color = C_RED if piece['army'] == 'rojo' else C_BLACK
                    # Pintamos la letra con su color
                    line += f"{color}{char}{C_RESET} "
                
                # 2. CASILLA DE AGUA
                elif terr == 'water':
                    line += f"{C_BLUE}={C_RESET} "
                
                # 3. CASILLA VACÍA
                else:
                    line += f"{C_GRAY}0{C_RESET} "
            
            # Número de fila (derecha) para referencia rápida
            print(line + f"{10-r}")

        print("   "+" ".join("abcdefghij")) # Cabecera Columnas

        # --- ESTADÍSTICAS (Reservas y Mazmorras) ---
        print("-" * 30)
        
        # Función auxiliar para imprimir listas bonitas
        def fmt_list(d, color_code):
            items = []
            # Ordenamos por tipo de pieza para consistencia
            for p_type, count in d.items():
                if count > 0:
                    char = TYPE_TO_CHAR.get(p_type, '?')
                    items.append(f"{char}:{count}")
            return f"{color_code}" + " ".join(items) + f"{C_RESET}"

        def fmt_dungeon(lst, color_code):
            if not lst: return f"{C_GRAY}(Vacía){C_RESET}"
            chars = [TYPE_TO_CHAR.get(p['type'], '?') for p in lst]
            return f"{color_code}" + "".join(chars) + f"{C_RESET}"

        # Imprimir Reservas
        print(f"📦 RESERVAS:")
        print(f"   🔴 Rojo:  {fmt_list(self.reserves['rojo'], C_RED)}")
        print(f"   ⚫ Negro: {fmt_list(self.reserves['negro'], C_BLACK)}")
        
        # Imprimir Mazmorras (Prisioneros capturados)
        print(f"⛓️  MAZMORRAS (Prisioneros):")
        # Nota: La mazmorra ROJA contiene piezas NEGRAS capturadas, y viceversa.
        print(f"   🔴 Tiene prisioneros: {fmt_dungeon(self.dungeons['rojo'], C_BLACK)}")
        print(f"   ⚫ Tiene prisioneros: {fmt_dungeon(self.dungeons['negro'], C_RED)}")
        print("-" * 30)

# Bloque 5. Parser y Ejecución.

In [1260]:
import re
from collections import deque

class CotadrezParser:
    def __init__(self, game): 
        self.game = game
        self.siege = False
        self.queue = deque()
        self.history = []
        self.march_active = False
        self.march_notation = ""
        
    def load_log(self, text):
        """Carga el log en la cola."""
        self.queue.clear()
        self.history.clear()
        
        text = text.replace("->ASEDIO", " ->ASEDIO ").replace("<-ASEDIO", " <-ASEDIO ")
        print("📥 Cargando log...")
        
        for line in text.strip().split('\n'):
            line = line.strip()
            if not line: continue
            
            if line.startswith("["): 
                self.history.append(line)  # <--- LÍNEA NUEVA: Guardamos el encabezado en el historial
                
                if "J1" in line: 
                    match = re.search(r'J1\s+["\']?([^"\'\]]+)', line)
                    if match: self.game.set_j1_color(match.group(1))
                continue
            
            if line.startswith("rF") or line == "//" or line == ">>>":
                self.queue.append(line)
                continue
            
            tokens = line.split()
            for t in tokens:
                if re.match(r'^\d', t) or "(" in t: continue
                self.queue.append(t)
                
        print(f"✅ Carga completa. {len(self.queue)} pasos listos.")

    def show_history(self):
        """Muestra el historial detallado."""
        if not self.history:
            print("\n📜 HISTORIAL VACÍO.")
            return
            
        print(f"\n📜 HISTORIAL DE LA PARTIDA ({len(self.history)} pasos):")
        for i, line in enumerate(self.history, 1):
            # Limpiamos colores ANSI para el historial si quieres texto plano, 
            # o los dejamos para que se vea bonito. Aquí los dejamos.
            print(f"{i:02}. {line}")
        print("-" * 30)

    def step(self):
        """Ejecuta UN paso y construye una línea de log unificada."""
        if not self.queue:
            print("⚠️ Fin del log.")
            return

        t = self.queue.popleft()
        if self.march_active:
            display_t = self.march_notation
        else:
            display_t = t
        turn_icon = self.game.get_icon()
        log_line = "" # <--- VARIABLE CENTRALIZADA

        has_more = "&" in t
        if has_more: self.march_notation = t

        # Es marcha forzada si hay '&' AHORA o si venimos de uno (march_active)
        if has_more or self.march_active:
            msg_suffix = " (⚡ Marcha Forzada)"
        else:
            msg_suffix = ""
            
        # Actualizamos la memoria para la SIGUIENTE vuelta
        self.march_active = has_more
        
        # Gestión de cola para acciones encadenadas (Marcha Forzada)
        if has_more:
            # Dividimos las órdenes (Ej: "M1&M2" -> ["M1", "M2"])
            subtokens = t.split('&')
            # Nos quedamos con la PRIMERA para ejecutarla ahora
            t = subtokens[0] 
            # Devolvemos el resto a la cola (al principio)
            for extra_token in reversed(subtokens[1:]):
                self.queue.appendleft(extra_token)

        # ---------------- LÓGICA DE PROCESAMIENTO ----------------
        
        # 1. FASES
        if t.startswith("rF"):
            cs = re.findall(r'([a-j](?:10|[0-9]))', t)
            if len(cs)==2: 
                r1,c1=self.game.coord_to_index(cs[0])
                r2,c2=self.game.coord_to_index(cs[1])
                coords_txt = f"{self.game.fmt(r1,c1)} : {self.game.fmt(r2,c2)}"
                log_line = f"[{t}] 🏗️  FORTALEZA establecida en {coords_txt}"
                self.game.deploy_fortress_area(r1,c1,r2,c2)
            else:
                log_line = f"[{t}] ❌ ERROR FORTALEZA"

        elif t == "//": 
            log_line = f"[{t}] 🔄 CAMBIO DE DESPLIEGUE"
            self.game.turn_color = self.game.j2_color
        
        elif t == ">>>": 
            log_line = f"[{t}] ⚔️  INICIO DEL COMBATE"
            self.game.game_phase = 'play'
            self.game.turn_color = self.game.j1_color

        # 2. ESTADOS
        elif "ASEDIO" in t: 
            self.siege = ("->" in t)
            log_line = f"[{t}] 🚨 ASEDIO: {self.siege}"
        
        # elif "&" in t: 
        #    log_line = f"[{t}] ⚡ Marcha Forzada"
        #    if self.game.game_phase == 'play': self.game.switch_turn()

        # 3. ACCIONES
        elif re.match(r'^m[A-Z]', t): 
            char = t[1]
            self.game.rescue_piece(PIECE_MAP.get(char, '?'))
            log_line = f"[{t}] 🚁 Rescate ({char})"
        
        else:
            # Movimiento Estándar
            m = re.match(r'^([r])?([A-Z])([a-j](?:10|[0-9]))?([x=])?([A-Z])?([a-j](?:10|[0-9]))', t)
            if m:
                pre, char, org, act, tgt, dst = m.groups()
                pt = PIECE_MAP.get(char)
                dr, dc = self.game.coord_to_index(dst)
                
                # A) DESPLIEGUE
                if pre == 'r': 
                    # Si devuelve True, es que se puso. Si devuelve None/False, falló.
                    if self.game.deploy_unit(pt, dr, dc):
                        log_line = f"[{t}] 🆕 Despliegue {pt.upper()} -> {self.game.fmt(dr, dc)}"
                    else:
                        log_line = f"[{t}] ❌ DESPLIEGUE FALLIDO (Terreno/Ocupado)"
                
                # B) MOVIMIENTO
                else:
                    or_r, or_c = None, None
                    error_msg = None
                    
                    if org: or_r, or_c = self.game.coord_to_index(org)
                    else:
                        cands = self.game.find_piece_candidates(pt, dr, dc, act=='x')
                        if len(cands) == 1: or_r, or_c = cands[0]['r'], cands[0]['c']
                        elif len(cands) == 0: error_msg = "❌ ILEGAL (Sin ruta/candidato)"
                        else: error_msg = "❌ AMBIGUO"
                    
                    if or_r is None and not error_msg: error_msg = "❌ ERROR COORD"

                    if error_msg:
                        log_line = f"[{t}] {error_msg}"
                    else:
                        coords_txt = f"{self.game.fmt(or_r, or_c)} -> {self.game.fmt(dr, dc)}"
                        
                        if act == 'x' and pt in ['trabuquete', 'escorpion']:
                            log_line = f"[{t}] 🔥 DISPARO {coords_txt}"
                            self.game.execute_shot(or_r, or_c, dr, dc)
                        elif act == '=':
                            log_line = f"[{t}] 🔄 RELEVO {coords_txt}"
                            self.game.apply_move_internal(or_r, or_c, dr, dc)
                        else:
                            verb = "CAPTURA" if act=='x' else "MUEVE"
                            # Primero definimos la línea como un INTENTO
                            log_line = f"[{display_t}] ➡️ {verb} {coords_txt}{msg_suffix}"
                            
                            # Ejecutamos y capturamos el resultado (True/False)
                            exito = self.game.apply_move_internal(or_r, or_c, dr, dc)
                            
                            # Si falló, añadimos una marca al texto del parser
                            if exito is False:
                                log_line += " (🚫 ANULADO POR MOTOR)"
            else:
                log_line = f"[{t}] ❓ TOKEN DESCONOCIDO"

        # ---------------- FINALIZACIÓN UNIFICADA ----------------
        
        if log_line:
            # Si el mensaje contiene una X de error, usamos el triangulo amarillo
            final_icon = "⚠️" if "❌" in log_line else turn_icon
            
            # Reemplazamos el token "[t]" por "[t] ICONO"
            full_log_line = log_line.replace(f"[{display_t}]", f"[{display_t}] {final_icon}", 1)
       
        # 1. Imprimir la línea bonita
        print(f"\n{full_log_line}")
        
        # 2. Guardar en historial
        self.history.append(full_log_line)
        
        # 3. Gestión automática de turno
        # AÑADIMOS: and t != ">>>" para evitar que cambie nada más empezar
        if self.game.game_phase == 'play' and not self.siege and "❌" not in log_line and t != ">>>" and not has_more: 
            self.game.switch_turn()
            
# --- EJECUCIÓN ---
juego = CotadrezGame()
parser = CotadrezParser(juego)

log = """
    [Event "Test Gemini"]
    [J2 "Rojo"]
    [J1 "Negro"]

    rFg2h3
    rPi3
    rBh4
    rTg4
    rDf4
    rCf3
    rLf2
    rHf1
    rAi1
    rEi2
    rRh1
    rHg1
    rMe3
    rMf5
    rMh5
    //
    rFc8d9
    rMd6
    rMg6
    rMe7
    rRd10
    rEc10
    rPe10
    rDb9
    rBe9
    rCb8
    rTe8
    rLb10
    rHc7
    rLd7
    >>>
    Aj2
    Cc5
    Pj5
    L=Hc7
    Hf1e1&Hg1f1
    Pf8
    rAi3 Cb8c5
    De5 Cb8
    Af3e4 Lc6

    Le9e8
    Lc2c3
    Axe5
    Txd5 ->ASEDIO
    Pe7
    mE
    rRh6
    <-ASEDIO
    """

parser.load_log(log)

📥 Cargando log...
⚙️  J1=NEGRO | J2=ROJO
✅ Carga completa. 52 pasos listos.


In [1261]:
parser.step()


[rFg2h3] ⚫ 🏗️  FORTALEZA establecida en (8,6|g2) : (7,7|h3)


In [1262]:
juego.render()


   a b c d e f g h i j
10 0 0 0 0 0 0 0 0 0 0 10
 9 0 0 0 = 0 0 = 0 0 0 9
 8 0 0 0 0 0 0 0 0 0 0 8
 7 0 = 0 0 0 0 0 0 = 0 7
 6 0 0 0 0 0 0 0 0 0 0 6
 5 0 0 0 0 0 0 0 0 0 0 5
 4 0 = 0 0 0 0 0 0 = 0 4
 3 0 0 0 0 0 0 F F 0 0 3
 2 0 0 0 = 0 0 F F 0 0 2
 1 0 0 0 0 0 0 0 0 0 0 1
   a b c d e f g h i j
------------------------------
📦 RESERVAS:
   🔴 Rojo:  R:1 F:1 M:3 L:3 A:3 C:3 P:2 H:3 D:1 E:2 B:1 T:1
   ⚫ Negro: R:1 M:3 L:3 A:3 C:3 P:2 H:3 D:1 E:2 B:1 T:1
⛓️  MAZMORRAS (Prisioneros):
   🔴 Tiene prisioneros: (Vacía)
   ⚫ Tiene prisioneros: (Vacía)
------------------------------


In [1263]:
parser.show_history()


📜 HISTORIAL DE LA PARTIDA (4 pasos):
01. [Event "Test Gemini"]
02. [J2 "Rojo"]
03. [J1 "Negro"]
04. [rFg2h3] ⚫ 🏗️  FORTALEZA establecida en (8,6|g2) : (7,7|h3)
------------------------------


In [1265]:
import time
import copy  # <--- IMPORTANTE: Necesario para clonar el estado

def grabar_partida(nombre_archivo="partida_cotadrez.txt"):
    print("🔴 GRABADOR DE COTADREZ (V3) - CON UNDO")
    print("   - Escribe jugadas (ej: rPe10, Pe9...)")
    print("   - Escribe 'z' o 'undo' para DESHACER la última jugada.")
    print("   - Escribe '#' para TERMINAR y guardar.")
    print("-" * 40)

    game_rec = CotadrezGame()
    parser_rec = CotadrezParser(game_rec)
    raw_log = []
    
    # Variables para el sistema de UNDO (1 nivel)
    backup_parser = None
    backup_log = []

    while True:
        # A. Procesar cola pendiente (Marcha Forzada)
        if parser_rec.queue:
            print("\n⚡ Procesando acción encadenada...")
            time.sleep(0.5)
        else:
            # B. Pedir entrada
            try:
                entrada = input(f"\n({len(raw_log)}) > ").strip()
            except KeyboardInterrupt:
                print("\n🛑 Interrumpido.")
                break

            if not entrada: continue

            # --- COMANDO UNDO / DESHACER ---
            if entrada.lower() in ['z', 'undo', 'deshacer']:
                if backup_parser is not None:
                    print("⏮️  Deshaciendo última acción...")
                    # 1. Restauramos el parser completo (incluye historia y cola)
                    parser_rec = backup_parser
                    # 2. Restauramos la referencia del juego (para que el render funcione)
                    game_rec = parser_rec.game
                    # 3. Restauramos el log de texto
                    raw_log = list(backup_log)
                    
                    game_rec.render()
                    continue
                else:
                    print("⚠️  No hay nada que deshacer (o ya estás en el inicio).")
                    continue

            # --- COMANDO FIN ---
            if entrada == '#':
                print("\n🏁 FIN DE LA PARTIDA. Guardando...")
                raw_log.append(entrada) 
                break
            
            # --- GESTIÓN DE ENCABEZADOS ---
            if entrada.startswith("["):
                if len(raw_log) == 0:
                    print("\n🔄 NUEVA PARTIDA DETECTADA: Reiniciando tablero...")
                    game_rec = CotadrezGame()
                    parser_rec = CotadrezParser(game_rec)
                    raw_log.append(entrada)
                    parser_rec.load_log(entrada)
                    
                    # Al reiniciar, borramos el backup antiguo
                    backup_parser = None
                    backup_log = []
                    continue
                else:
                    print(f"ℹ️  Metadato registrado: {entrada}")
                    raw_log.append(entrada)
                    parser_rec.load_log(entrada)
                    continue

            # --- PREPARAR UNDO (SNAPSHOT) ---
            # Antes de meter una jugada nueva en la cola, guardamos el estado actual.
            # Solo guardamos si NO estamos en mitad de una cadena automática.
            if not parser_rec.queue:
                backup_parser = copy.deepcopy(parser_rec) # Copia profunda de TODO (Tablero incluido)
                backup_log = list(raw_log)                # Copia de la lista de textos

            # Añadir a cola
            parser_rec.queue.append(entrada)

        # C. EJECUTAR JUGADA
        hist_len_before = len(parser_rec.history)
        parser_rec.step()
        
        # D. VERIFICAR SI FUE VÁLIDA
        if len(parser_rec.history) > hist_len_before:
            last_line = parser_rec.history[-1]
            
            if "❌" in last_line or "🚫" in last_line:
                print("⚠️  Movimiento inválido. No se guardará.")
                # Nota: Si el usuario quiere borrar el mensaje de error del historial visual, 
                # puede pulsar 'z' y volverá al estado limpio previo al error.
            else:
                if not parser_rec.queue and entrada not in raw_log and not entrada.startswith("["):
                    # Verificación extra: No guardar comandos 'undo' o 'z' si se colaron
                    if entrada.lower() not in ['z', 'undo', 'deshacer']:
                        raw_log.append(entrada)
                
                if not entrada.startswith("["):
                    game_rec.render()

    # --- GUARDADO ---
    if raw_log:
        try:
            with open(nombre_archivo, "w", encoding="utf-8") as f:
                f.write("\n".join(raw_log))
            print(f"\n✅ Guardado en '{nombre_archivo}' ({len(raw_log)} líneas).")
        except Exception as e:
            print(f"❌ Error al guardar: {e}")

# --- EJECUCIÓN ---
grabar_partida()

🔴 GRABADOR DE COTADREZ (V3) - CON UNDO
   - Escribe jugadas (ej: rPe10, Pe9...)
   - Escribe 'z' o 'undo' para DESHACER la última jugada.
   - Escribe '#' para TERMINAR y guardar.
----------------------------------------



🔄 NUEVA PARTIDA DETECTADA: Reiniciando tablero...
📥 Cargando log...
✅ Carga completa. 0 pasos listos.
ℹ️  Metadato registrado: [J1 Rojo]
📥 Cargando log...
⚙️  J1=ROJO | J2=NEGRO
✅ Carga completa. 0 pasos listos.

[rFg2h3 rPi3 rBh4 rTg4 rDf4 rCf3 rLf2 rEf1 rAi1 rEi2 rRh1 rHg1 rMe3 rMf5 rMh5 //] ⚠️ ❌ ERROR FORTALEZA
⚠️  Movimiento inválido. No se guardará.

🏁 FIN DE LA PARTIDA. Guardando...

✅ Guardado en 'partida_cotadrez.txt' (3 líneas).
